# Tutorial 6: Multi-Agent Systems

In tutorial 4 you built one agent. Here you build **teams** of agents.

By the end you will be able to:

- say when one agent is enough and when a team helps;
- build a **pipeline**, where you fix the order;
- build a **supervisor**, where one agent gives work to others;
- build **handoffs**, where agents pass the conversation to each other;
- choose between the three by asking one question: **who decides what runs next?**

**Before you start:** tutorial 4, sections 2 to 5.

## 0. Mental model: a team of people

<img src="images/multi_agent_cards/One_agent_or_a_team.png" alt="One agent or a team card: a single robot struggles with calls, writing, calculating and reports; a team of four gives each robot one of those jobs. Start with one. Split the work when it struggles." width="650">

Imagine one employee who must research, write, check, answer the phone and do the accounts. Their instructions are ten
pages long and they make mistakes. Agents are the same: **many tools and a long prompt make an agent worse.**

A team splits the work. Each agent gets a short prompt, a few tools, and only the information its own job needs. There
are three common ways to organise the team:

| Pattern | Everyday picture | Who decides what runs next? | Section |
|---|---|---|---|
| **Pipeline** | An assembly line | You, in advance | 2 |
| **Supervisor** | A manager with specialists | The manager agent | 3 |
| **Handoffs** | A phone call being transferred | The agent holding the call | 4 |

<img src="images/multi_agent_cards/Who_decides_what_runs_next.png" alt="Who decides what runs next card: in a pipeline you decide, drawn as an assembly line; with a supervisor the manager agent decides, delegating to two workers; with handoffs the active agent decides, passing the phone from reception to a specialist." width="650">

```text
Pipeline        A → B → C

Supervisor          ┌→ A
                S ──┤         (A and B report back to S)
                    └→ B

Handoffs        A ──transfer──→ B   (B now talks to the user)
```

A team costs more: more model calls, so more time and money. **Stay with one agent until it visibly struggles.**

## 1. Setup

The same environment as tutorial 4:

```bash
pip install "langchain>=1.0" langchain-core langchain-groq python-dotenv langgraph
```

and the same `.env` with `GROQ_API_KEY` beside this notebook. If you hit a rate limit, wait a minute and run the cell again.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found, see tutorial 3, section 1."

from typing import TypedDict
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.graph import StateGraph, START, END

model = ChatGroq(model="openai/gpt-oss-20b", temperature=0, max_retries=6)
print("ready")

### Three small helpers

<img src="images/multi_agent_cards/Three_small_helpers.png" alt="Three small helpers card: ask posts a question into the agent, answer picks the final reply from the messages, show_trace lights up every step, and a recursion_limit stop sign blocks an endless loop." width="650">

`ask` sends one question to an agent, `answer` picks out its final reply, and `show_trace` (from tutorial 4) prints every
step. `recursion_limit` makes sure that no agent loops forever.

In [ ]:
def ask(agent, text):
    return agent.invoke({"messages": [HumanMessage(text)]}, config={"recursion_limit": 25})

def answer(result):
    return result["messages"][-1].content

def show_trace(result):
    for m in result["messages"]:
        who = m.name or type(m).__name__
        for call in getattr(m, "tool_calls", []):
            print(f"{who:>18}: → {call['name']}({call['args']})")
        if m.content:
            print(f"{who:>18}: {m.content}")

### Two tools

<img src="images/multi_agent_cards/Two_tools.png" alt="Two tools card: get_population is an atlas that turns Paris into 2.1M and multiply is a calculator showing 2.1M x 3 = 6.3M; both are marked as tested. Test tools first." width="650">

Sections 2 and 3 share these tools. The populations are rounded examples.

In [ ]:
POPULATION = {"tehran": 9_000_000, "paris": 2_100_000, "tokyo": 14_000_000}

@tool
def get_population(city: str) -> int:
    """Return the approximate population of a city, or 0 if it is unknown."""
    return POPULATION.get(city.lower(), 0)

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers together."""
    return a * b

QUESTION = "What is three times the population of Paris?"

**An agent is a tool-using model with one job.** Always test the tools first, since no agent can be better than its
tools:

In [ ]:
print(get_population.invoke({"city": "Paris"}))
print(multiply.invoke({"a": 3, "b": 2_100_000}))

## 2. Pipeline: the assembly line

We make a small newsroom. A **researcher** collects facts, a **writer** turns them into a paragraph, and a **critic**
accepts the paragraph or sends it back:

```text
START → research → write → review → END
                     ▲        │
                     └────────┘  not good enough: try again
```

You know the order in advance, so **you** write it as edges.

### 2.1 Agent 1: the researcher

<img src="images/multi_agent_cards/The_researcher.png" alt="The researcher card: asked for three times the population of Paris, the researcher uses an atlas and a calculator and returns notes saying Paris is about 2.1M and three times that is about 6.3M. Question, tools, notes." width="650">

Its job is to find facts, so it is the only one with tools.

In [ ]:
researcher = create_agent(
    model,
    tools=[get_population, multiply],
    system_prompt=(
        "You are a researcher. Use your tools to collect every fact the question needs. "
        "Reply with short bullet points only, and include the figures you looked up."
    ),
    name="researcher",
)

Test it alone:

In [ ]:
result = ask(researcher, QUESTION)
show_trace(result)

notes = answer(result)

It looked up the population, multiplied, and wrote notes. We keep the notes: they are what the next agent needs.

### 2.2 Agent 2: the writer

<img src="images/multi_agent_cards/The_writer.png" alt="The writer card: the writer takes a note saying 6.3M and types: three times the population of Paris is about 6.3 million. Notes in. Draft out." width="650">

Its job is to write. It has **no tools**, so it cannot look anything up, and it must work from the notes.

In [ ]:
writer = create_agent(
    model,
    system_prompt=(
        "You are a writer. Turn the notes into one clear sentence that answers the question. "
        "Use only facts from the notes. If there is editor feedback, apply it."
    ),
    name="writer",
)

Test it alone, with the researcher's notes:

In [ ]:
draft = answer(ask(writer, f"Question: {QUESTION}\nNotes: {notes}"))
print(draft)

### 2.3 Agent 3: the critic

<img src="images/multi_agent_cards/The_critic.png" alt="The critic card: compared with the fact 6.3M, a draft of 6.3M gets approved True, while a draft of 9M gets approved False with the feedback: use 6.3M." width="650">

Its job is to decide: accept or reject. The program must act on that decision, so we want a real `True` or `False` and
not a sentence. That is `response_format` from tutorial 4, section 5.5.

First, the form the critic fills in:

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy

class Review(BaseModel):
    approved: bool = Field(description="True if the paragraph is correct and answers the question.")
    feedback: str = Field(description="What to fix, or an empty string if approved.")

Then the agent that fills it in:

In [ ]:
critic = create_agent(
    model,
    system_prompt=(
        "You are an editor. Approve the paragraph if it answers the question and its numbers "
        "agree with the notes. Otherwise reject it and say briefly what to fix."
    ),
    response_format=ToolStrategy(Review),
    name="critic",
)

Test it alone, **twice**: once with the real draft and once with a deliberately wrong one. A critic that approves
everything is useless, and you only find out by trying a bad input.

In [ ]:
def judge(paragraph):
    prompt = f"Question: {QUESTION}\nNotes: {notes}\nParagraph: {paragraph}"
    return ask(critic, prompt)["structured_response"]

print("real draft: ", judge(draft))
print("wrong draft:", judge("Three times the population of Paris is 9 million."))

All three agents work. Now we connect them.

### 2.4 The shared state

<img src="images/multi_agent_cards/The_shared_state.png" alt="The shared state card: the researcher, writer and critic pin notes, draft, attempts, approved and feedback on one shared whiteboard below the question." width="650">

Agents in a pipeline talk through a **state**, the whiteboard from tutorial 4. Each agent reads some keys and writes
others. Decide this before writing any graph code:

| Step | Reads | Writes |
|---|---|---|
| research | `question` | `notes` |
| write | `question`, `notes`, `feedback` | `draft`, `attempts` |
| review | `question`, `notes`, `draft` | `approved`, `feedback` |

In [ ]:
class NewsState(TypedDict):
    question: str
    notes: str
    draft: str
    approved: bool
    feedback: str
    attempts: int

### 2.5 One node per agent

<img src="images/multi_agent_cards/One_node_per_agent.png" alt="One node per agent card: the wrapper reads question, notes and feedback from the state, carries a prompt to the writer, and brings draft and attempts back to the state. Build prompt, ask agent, return changed keys." width="650">

A node is a small **wrapper** around an agent. All three do the same three things: build a prompt from the state, ask
the agent, and return the keys that changed.

In [ ]:
def research(state: NewsState) -> dict:
    result = ask(researcher, state["question"])
    return {"notes": answer(result)}

In [ ]:
def write(state: NewsState) -> dict:
    prompt = (
        f"Question: {state['question']}\n"
        f"Notes: {state['notes']}\n"
        f"Editor feedback: {state['feedback']}"
    )
    result = ask(writer, prompt)
    return {"draft": answer(result), "attempts": state["attempts"] + 1}

In [ ]:
def review(state: NewsState) -> dict:
    prompt = (
        f"Question: {state['question']}\n"
        f"Notes: {state['notes']}\n"
        f"Paragraph: {state['draft']}"
    )
    verdict = ask(critic, prompt)["structured_response"]
    return {"approved": verdict.approved, "feedback": verdict.feedback}

A node is an ordinary function, so you can test it without a graph. Just hand it a dictionary:

In [ ]:
print(write({"question": QUESTION, "notes": notes, "feedback": "Start with the words 'In total'.", "attempts": 0}))

### 2.6 The router

<img src="images/multi_agent_cards/The_router.png" alt="The router card: a signal operator sends the train to the writer when the draft is rejected and under 3 attempts, and to done when it is approved or 3 attempts were made." width="650">

After the review there are two ways to go. The router looks at the state and names one of them.

In [ ]:
MAX_ATTEMPTS = 3

def after_review(state: NewsState) -> str:
    if state["approved"]:
        return "done"
    if state["attempts"] >= MAX_ATTEMPTS:
        return "done"
    return "revise"

print(after_review({"approved": True, "attempts": 1}))
print(after_review({"approved": False, "attempts": 1}))
print(after_review({"approved": False, "attempts": 3}))

The second `if` is the safety limit: a critic that is never satisfied cannot keep the loop running forever.

### 2.7 Combine: the newsroom

<img src="images/multi_agent_cards/The_newsroom.png" alt="The newsroom card: a conveyor runs through research, write and review to the exit when done, with a revise arrow from review back to write. You set the order." width="650">

Every part exists and has been tested. Combining is only nodes and edges, and it copies the drawing at the top of this
section.

In [ ]:
builder = StateGraph(NewsState)

# the workers
builder.add_node("research", research)
builder.add_node("write", write)
builder.add_node("review", review)

# the assembly line
builder.add_edge(START, "research")
builder.add_edge("research", "write")
builder.add_edge("write", "review")

# the way back
builder.add_conditional_edges("review", after_review, {"revise": "write", "done": END})

newsroom = builder.compile()

Run it, and watch each node write to the whiteboard. `stream_mode="updates"` gives us what each node returned:

In [ ]:
start = {"question": QUESTION, "feedback": "none", "attempts": 0}

for step in newsroom.stream(start, stream_mode="updates"):
    for node, update in step.items():
        print(f"{node:>9}: {update}")

### 2.8 Does the loop work?

<img src="images/multi_agent_cards/Test_the_revision_loop.png" alt="Test the revision loop card: four panels show a wrong draft of 9M, feedback saying to use 6.3M instead, a fixed draft of 6.3M, and an approval." width="650">

The draft was probably accepted at once, so the backwards edge was never used. **Untested code is not working code**, so
let us force a rejection: replace the writer node with one whose first draft is wrong.

In [ ]:
def careless_write(state: NewsState) -> dict:
    if state["attempts"] == 0:
        return {"draft": "Three times the population of Paris is 9 million.", "attempts": 1}
    return write(state)

builder = StateGraph(NewsState)
builder.add_node("research", research)
builder.add_node("write", careless_write)
builder.add_node("review", review)
builder.add_edge(START, "research")
builder.add_edge("research", "write")
builder.add_edge("write", "review")
builder.add_conditional_edges("review", after_review, {"revise": "write", "done": END})

for step in builder.compile().stream(start, stream_mode="updates"):
    for node, update in step.items():
        print(f"{node:>9}: {update}")

Now you can see the full cycle: a wrong draft, a rejection with feedback, a better draft, an approval.

**What to remember about pipelines**

- **You** decide the order, so the behaviour is predictable and easy to debug.
- Each agent sees **only what its wrapper passes in**. The writer never saw the researcher's tool calls.
- Use structured output wherever the **program** must act on an agent's answer.
- At the end, check `approved`. The loop may have stopped because of the limit.

## 3. Supervisor: the manager

A pipeline runs every step for every question. Sometimes the steps **depend on the question**: one needs a lookup, one
needs arithmetic, one needs both. Then a model should choose:

```text
                  ┌→ data_agent ──┐
user → supervisor ┤               ├→ supervisor → answer
                  └→ math_agent ──┘
```

The trick is short: **wrap each worker agent as a tool, and give the tools to another agent.**

### 3.1 Worker 1: the data agent

<img src="images/multi_agent_cards/The_data_agent.png" alt="The data agent card: asked for the population of Tokyo, the agent looks in its get_population atlas and replies 14,000,000. Look up a population." width="650">

In [ ]:
data_agent = create_agent(
    model,
    tools=[get_population],
    system_prompt="You look up city populations with your tool. Reply with the figure and nothing else.",
    name="data_agent",
)

show_trace(ask(data_agent, "Population of Tokyo?"))

### 3.2 Worker 2: the math agent

<img src="images/multi_agent_cards/The_math_agent.png" alt="The math agent card: given 14,000,000 x 2, the agent uses its multiply calculator and replies 28,000,000. Send every number in the task." width="650">

In [ ]:
math_agent = create_agent(
    model,
    tools=[multiply],
    system_prompt="You do arithmetic with your tool. Reply with the result and nothing else.",
    name="math_agent",
)

show_trace(ask(math_agent, "What is 14000000 times 2?"))

### 3.3 Turn each worker into a tool

<img src="images/multi_agent_cards/Agents_inside_tools.png" alt="Agents inside tools card: a toolbox holds ask_data_agent and ask_math_agent, each a booth with an agent inside, next to a docstring describing them; 14,000,000 x 2 goes in and 28,000,000 comes out. Task in. Final reply out." width="650">

A tool is a function with a docstring. So we write a function that asks the worker and returns its answer:

In [ ]:
@tool
def ask_data_agent(task: str) -> str:
    """Look up the population of a city."""
    return answer(ask(data_agent, task))

@tool
def ask_math_agent(task: str) -> str:
    """Do arithmetic. Include every number in the task, because this agent sees nothing else."""
    return answer(ask(math_agent, task))

Test them as tools. From the outside you cannot tell that there is a whole agent inside:

In [ ]:
print(ask_data_agent.invoke({"task": "Population of Tehran?"}))
print(ask_math_agent.invoke({"task": "Multiply 9000000 by 2"}))

**The docstring is for the supervisor.** It is how it learns who to send the work to. "Include every number" is there
because the worker sees only the `task` string. It cannot see the conversation, so "multiply it by three" would mean
nothing to it.

### 3.4 The supervisor

<img src="images/multi_agent_cards/The_supervisor.png" alt="The supervisor card: a manager sends tasks to the data and math booths, receives their results, and gives the user the reply. Delegate. Wait. Reply." width="650">

An ordinary agent. Its tools just happen to be agents.

In [ ]:
supervisor = create_agent(
    model,
    tools=[ask_data_agent, ask_math_agent],
    system_prompt=(
        "You coordinate two specialists. Delegate one step at a time and wait for the result "
        "before the next step. Never recall or compute figures yourself. "
        "Finish with one short sentence, in plain text."
    ),
    name="supervisor",
)

### 3.5 Run the team

<img src="images/multi_agent_cards/Run_the_team.png" alt="Run the team card: three times the population of Paris needs data, then math, and gives 6.3M; the population of Tokyo needs data only and gives 14M." width="650">

A question that needs **both** workers:

In [ ]:
show_trace(ask(supervisor, QUESTION))

A question that needs **only one**:

In [ ]:
show_trace(ask(supervisor, "What is the population of Tokyo?"))

The same team took two different paths, and no edge was written. The supervisor chose.

The workers' own steps (`get_population`, `multiply`) are missing from the trace. They happened inside the tools. The
supervisor saw only the final replies, which keeps its context short.

**What to remember about supervisors**

- A worker is **an agent inside a tool**.
- Context is separated in both directions: the worker sees only its task, the supervisor only the result.
- Workers never talk to the user. The supervisor does.
- Flexible, at the price of extra model calls for the supervisor itself.

## 4. Handoffs: transferring the call

You call a helpdesk. The receptionist listens, says "I will transfer you to the exam office", and **leaves the call**. The
specialist answers you directly. There is no manager in the middle:

```text
START → triage ──transfer_to_grades──→ grades → END
           │
           └─────transfer_to_exams───→ exams  → END
```

### 4.1 Specialist 1: grades

<img src="images/multi_agent_cards/The_grades_specialist.png" alt="The grades specialist card: the specialist looks up s101 with get_grade and replies that the midterm grade is 18.5. Student ID to grade." width="650">

In [ ]:
GRADES = {"s101": 18.5, "s102": 16.0}

@tool
def get_grade(student_id: str) -> str:
    """Return a student's midterm grade, given an id such as s101."""
    return str(GRADES.get(student_id.lower(), "no grade on record"))

grades = create_agent(
    model,
    tools=[get_grade],
    system_prompt="You answer questions about grades. Use your tool, then reply to the student in one sentence.",
    name="grades",
)

show_trace(ask(grades, "I am s101. What is my grade?"))

### 4.2 Specialist 2: exams

<img src="images/multi_agent_cards/The_exams_specialist.png" alt="The exams specialist card: the specialist looks up the LLM course with get_exam_date and replies that the final exam is on 2026-12-20. Course to exam date." width="650">

In [ ]:
EXAM_DATES = {"llm": "2026-12-20", "nlp": "2026-12-27"}

@tool
def get_exam_date(course: str) -> str:
    """Return the final exam date of a course, given its short name such as llm."""
    return EXAM_DATES.get(course.lower(), "no exam on record")

exams = create_agent(
    model,
    tools=[get_exam_date],
    system_prompt="You answer questions about exam dates. Use your tool, then reply to the student in one sentence.",
    name="exams",
)

show_trace(ask(exams, "When is the LLM exam?"))

### 4.3 The transfer button

<img src="images/multi_agent_cards/The_transfer_button.png" alt="The transfer button card: reception presses Grades and passes the phone with the tool request and its matching receipt, using goto, Command.PARENT and update messages. Pass control and the conversation." width="650">

This is the only new idea in the section, so we take it slowly. A normal tool **returns a result** to its own agent. A
transfer tool **moves control** to another agent. It does so by returning a `Command` (you met it in tutorial 4, section
5.8) with three parts:

| Part | Meaning | In the phone call |
|---|---|---|
| `goto="grades"` | Run this node next | Dial the grades office |
| `graph=Command.PARENT` | The node is in the **outer** graph, outside this agent | Leave your own desk |
| `update={"messages": [...]}` | What to add to the shared conversation | Pass on the notes |

The notes must contain **two** messages that belong together: the triage agent's **request** ("call the transfer tool")
and the **receipt** that answers it. Model APIs reject a tool call that has no answer.

In [ ]:
from langchain.tools import ToolRuntime
from langgraph.types import Command

def transfer(target: str, runtime: ToolRuntime) -> Command:
    request = runtime.state["messages"][-1]
    receipt = ToolMessage(
        f"Transferred to {target}.",
        name=f"transfer_to_{target}",
        tool_call_id=runtime.tool_call_id,
    )
    return Command(
        goto=target,
        graph=Command.PARENT,
        update={"messages": [request, receipt]},
    )

Now each button is a two-line tool:

In [ ]:
@tool
def transfer_to_grades(runtime: ToolRuntime) -> Command:
    """Transfer the student to the grades specialist."""
    return transfer("grades", runtime)

@tool
def transfer_to_exams(runtime: ToolRuntime) -> Command:
    """Transfer the student to the exam schedule specialist."""
    return transfer("exams", runtime)

print("the model sees these arguments:", transfer_to_grades.tool_call_schema.model_json_schema()["properties"])

Empty. `runtime` is filled in by LangChain and hidden from the model, which only decides **whether** to press the button.

### 4.4 The receptionist

<img src="images/multi_agent_cards/The_receptionist.png" alt="The receptionist card: for an exam question the receptionist presses the Exams button; for a good morning it replies itself. Transfer once, or answer here." width="650">

Triage has no knowledge of grades or exams, just the two buttons.

In [ ]:
triage = create_agent(
    model,
    tools=[transfer_to_grades, transfer_to_exams],
    system_prompt=(
        "You are the receptionist of a course helpdesk. If the question is about grades or exam "
        "dates, make exactly one transfer. Otherwise answer briefly yourself."
    ),
    name="triage",
)

We can test triage alone only with a question it answers itself. The buttons need an outer graph to lead somewhere, and
we have not built it yet.

In [ ]:
show_trace(ask(triage, "Hello! What can you help me with?"))

### 4.5 Combine: the helpdesk

<img src="images/multi_agent_cards/The_helpdesk.png" alt="The helpdesk card: triage passes the phone to grades under a Command flag, with exams alongside, and one MessagesState ribbon runs across all three desks. One shared conversation." width="650">

Agents are graphs, and a graph can be a node. Every agent shares the same conversation, so the state is just a list of
messages. `MessagesState` is that state, ready-made.

In [ ]:
from langgraph.graph import MessagesState

builder = StateGraph(MessagesState)

# the three agents, used directly as nodes
builder.add_node("triage", triage, destinations=("grades", "exams"))
builder.add_node("grades", grades)
builder.add_node("exams", exams)

# every call starts at the reception
builder.add_edge(START, "triage")

# after a specialist answers, we are done
builder.add_edge("grades", END)
builder.add_edge("exams", END)

helpdesk = builder.compile()

There is **no edge** from `triage` to the specialists. The `Command` does that at run time. `destinations` only helps
when drawing the graph, and it does not route anything.

### 4.6 Run the helpdesk

<img src="images/multi_agent_cards/Run_the_helpdesk.png" alt="Run the helpdesk card: an LLM final question is answered by exams, a midterm grade question by grades, and a good morning by triage. Whoever holds the call answers." width="650">

Three callers, three paths.

In [ ]:
show_trace(ask(helpdesk, "When is the final exam of the LLM course?"))

In [ ]:
show_trace(ask(helpdesk, "I am student s102. What is my midterm grade?"))

In [ ]:
show_trace(ask(helpdesk, "Good morning! Who are you?"))

Look at **who speaks last**: `exams`, then `grades`, then `triage`. Whoever holds the call answers the user.

**What to remember about handoffs**

- The **active agent** decides whether to pass control.
- The next agent sees the **shared conversation** and talks to the user directly.
- `Command(goto=..., graph=Command.PARENT)` moves control. The request and its receipt travel together.
- "Exactly one transfer" is in the prompt because two transfers would send the call to two places at once.

## 5. Choosing a pattern

| | Pipeline | Supervisor | Handoffs |
|---|---|---|---|
| Who decides? | You | The supervisor | The active agent |
| Agents are | Nodes in wrappers | Tools | Nodes |
| What an agent sees | What the wrapper passes | Only its task | The shared conversation |
| Who answers the user? | The last step | The supervisor | Whoever holds the call |
| Best for | Fixed steps: research, write, check | Steps that depend on the question | Conversations with specialists |
| Main risk | Rigid | Extra cost, vague delegation | Harder to predict and debug |

Three questions settle most cases:

1. **Can one agent do it?** Then use one agent.
2. **Do you know the steps in advance?** Pipeline.
3. **If not, should a specialist talk to the user directly?** Yes: handoffs. No: supervisor.

Patterns mix. A supervisor's tool may call a whole pipeline, since a compiled graph is called like an agent.

## 6. Exercises

1. **Pipeline.** Add a `translate` agent that turns the approved draft into Persian. Test it alone, then add a node and
   change the `"done"` destination.
2. **Pipeline.** Make `careless_write` wrong every time. How many attempts run, and what is `approved` at the end?
3. **Supervisor.** Add a third worker with a tool that returns the area of a city. Ask for people per square kilometre.
   Which docstrings did you have to think about?
4. **Supervisor.** Remove "Include every number" from the docstring. Does delegation still work?
5. **Handoffs.** Add a `projects` specialist that knows the project deadline.
6. **Choose.** For each, name a pattern and give a reason: (a) translate, then proofread; (b) a travel assistant that may
   need flights, hotels or both; (c) a shop where sales passes you to repairs.

## Glossary

| Term | Meaning |
|---|---|
| **Multi-agent system** | Several agents, each with its own prompt and tools, working on one task. |
| **Pipeline** | Agents as graph nodes, in an order fixed by your edges. |
| **Wrapper node** | A function that builds an agent's prompt from the state and stores its answer. |
| **Router** | A function that reads the state and names the next step. |
| **Supervisor** | An agent whose tools are other agents. |
| **Worker** | An agent that does one task for a supervisor. |
| **Context isolation** | Each agent sees only the information it was given. |
| **Handoff** | Passing control, and the conversation, to another agent. |
| **`Command`** | A return value that updates the state and may choose the next node. |
| **`Command.PARENT`** | Says that the target node is in the outer graph. |
| **`ToolRuntime`** | Gives a tool access to state and context, hidden from the model. |
| **`MessagesState`** | A ready-made state with a single `messages` list. |